# 🔭 15 · Discovering the unknown: template-free detection and de novo structural pattern mining
### Finding complexes nobody told you to look for: invariant features, Markov random fields and Multi-Pattern Pursuit

**BiologyToResearch** · Part C — Classical computational cryo-ET (2009–2019) · Module 15 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/15_template_free_discovery.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier, CPU only (about 4–6 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn, scikit-image |
| **Prerequisites** | Module 02 (rotation-invariant features), 04 (clustering, Wilcoxon test), 10 (FSC, SSNR), 14 (averaging, classification, the flatland aligner) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Xu, Beck & Alber 2011 (template-free detection); Xu *et al.* 2019 (de novo structural pattern mining, MPP) — main; Wu *et al.* 2019 (book chapter: template-based vs template-free); Frazier, Xu & Alber 2017 (TomoMiner). See [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods. They illustrate the core idea of each paper;
> they do **not** reproduce the papers' numbers. Section 2 works on a small simulated 3D tomogram; Section 3 uses the
> **2D analogue** ("flatland") of subtomograms from Module 14 so that hundreds of alignments run in seconds. **All
> molecules and data are generated by code** (nothing is downloaded): realistic stand-ins with the size and symmetry
> of the PDB entries they are named after.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Contrast** template-based, supervised and template-free strategies for "visual proteomics", and say what each can
   and cannot discover.
2. **Implement** template-free candidate detection (local maxima + watershed) and rotation-invariant voxel features
   (spherical-harmonic energy per shell), and **verify** their invariance.
3. **Cluster** detected objects by the similarity of their voxel-feature content, and **explain** how a Markov random
   field whose neighbourhood includes *feature-space* neighbours lets distant copies of a complex correct each other.
4. **Compute** the spectral SNR of a set of aligned particles, convert it to $\rho=\eta/(2+\eta)$ and the pattern score
   SFSC, and **explain** why SFSC favours large, homogeneous sets.
5. **Implement** a mini Multi-Pattern Pursuit: over-clustering, greedy disjoint selection by SFSC, re-alignment and
   Wilcoxon signed-rank redundancy removal, iterated to stability.
6. **Summarise** what de novo pattern mining revealed in bacterial cells, and why rare complexes are the hardest to find.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools and molecules | 5 min |
| 1 · Templates or not? | three strategies for visual proteomics | 10 min |
| 2 · Template-free detection (2011) | candidates, invariant features, object clustering, an MRF | 50 min |
| 3 · Multi-Pattern Pursuit (2019) | SSNR → SFSC, selection, redundancy removal, a mini-MPP | 55 min |
| 4 · What pattern mining bought biology | findings in three bacteria | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 20 min |

## 🧫 The biological question

A bacterial proteome contains a few thousand kinds of proteins, and hundreds of them assemble into complexes large
enough to see in a tomogram. Template matching (Module 11) can only find the handful whose structures we already know and
think to search for. But some of the most interesting objects in a cell are the ones nobody expected: a complex that
exists only in situ, an unusual assembly on a membrane, a known machine in an unknown state. **Can a computer list the
recurring 3D shapes in a cell, and where they are, without being told what to look for?** That is *de novo* (from
scratch) visual proteomics, the goal of the two Xu-lab papers at the heart of this notebook.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected (fine: this notebook only needs a CPU).")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("skimage", "scikit-image")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, sparse
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import squareform
from scipy.stats import wilcoxon
from skimage.segmentation import watershed
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors

SEED = 0
rng = np.random.default_rng(SEED)
DEVICE = "cpu"                             # no deep learning here; kept for course-wide consistency
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED, "| device", DEVICE)

### 🧰 Course toolkit

Copied verbatim from [`course_toolkit.py`](../course_toolkit.py): the molecular zoo, rotations, Fourier masks, noise,
the crowded-cell packer and the fast tomogram simulator of Module 09. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0


def make_crowded_cell(templates: dict, counts: dict, shape=(48, 160, 160), rng=None,
                      membrane: bool = True, tries_per_particle: int = 400):
    """Pack randomly rotated templates into a volume without overlap (bounding spheres).

    A simple rejection-sampling stand-in for the packing used by Pei et al. (2016)
    and Liu et al. (2020).  Particles are placed in random class order so that every
    class gets a fair chance.  Returns (clean_volume, particles) where each particle
    is dict(cls, center_zyx, R).  Optionally adds a spherical membrane vesicle.
    """
    rng = rng or np.random.default_rng(0)
    vol = np.zeros(shape, np.float32)
    occ_c, occ_r, particles = [], [], []
    if membrane:  # a vesicle: thin shell, ~4 nm thick lipid bilayer at 1 nm/voxel
        zc, yc, xc = shape[0] / 2, shape[1] * 0.78, shape[2] * 0.22
        rad = min(shape[1], shape[2]) * 0.16
        g = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
        d = np.sqrt((g[0] - zc)**2 + (g[1] - yc)**2 + (g[2] - xc)**2)
        vol += 0.6 * np.exp(-((d - rad)**2) / (2 * 1.5**2))
        occ_c.append(np.array([zc, yc, xc])); occ_r.append(rad + 3)
    radii = {cls: support_radius(templates[cls], level=0.35) for cls in counts}
    order = [cls for cls, n in counts.items() for _ in range(n)]
    rng.shuffle(order)
    for cls in order:
        t, rad_t = templates[cls], radii[cls]
        for _ in range(tries_per_particle):
            c = np.array([rng.uniform(rad_t, s - rad_t) if s > 2 * rad_t else s / 2 for s in shape])
            if any(np.linalg.norm(c - oc) < rad_t + orr for oc, orr in zip(occ_c, occ_r)):
                continue
            R = random_rotation(rng)
            sub = rotate_volume(t, R)
            h = np.array(sub.shape) // 2
            lo = np.round(c).astype(int) - h
            sl_v, sl_s = [], []
            for a in range(3):
                a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
                sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
            vol[tuple(sl_v)] += sub[tuple(sl_s)]
            occ_c.append(c); occ_r.append(rad_t)
            particles.append(dict(cls=cls, center_zyx=c, R=R))
            break
    return vol, particles


def simulate_tomogram(clean: np.ndarray, rng, snr: float = 0.05, tilt_max_deg: float = 60.0,
                      ctf_pixel_A: float | None = 10.0, defocus_um: float = 5.0) -> np.ndarray:
    """Fast tomogram simulation: (optional 3D CTF-like filter) → missing wedge → noise.

    The CTF is applied isotropically here (a simplification of per-tilt CTF).
    """
    v = clean
    if ctf_pixel_A:
        ctf = ctf_2d(clean.shape, ctf_pixel_A, defocus_um=defocus_um, kV=300, B=0)
        v = apply_fourier_mask(v, -ctf)          # sign flip: dense objects appear bright
    v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng)
print("🧰 toolkit loaded")

---
## 1 · Templates or not? Three strategies for visual proteomics

> 📍 **Where we are.** Module 11 found particles with templates; Modules 12–14 aligned, averaged and classified them,
> mostly with references we supplied. This module removes the last crutch: **no template at all**. Module 22 will do the
> same with deep learning (DISCA), and Module 33 will turn the discovered patterns into biology.

Wu, Zeng, Zhu, Gao & Xu (2019) organise cryo-ET "structural pattern mining" into three families:

| Strategy | Needs | Finds | Main risk |
|---|---|---|---|
| **Template-based** (template matching, Module 11; reference-based averaging) | a known structure for each target | copies of what you already know | reference bias (Module 14): sees what it is told to see; misses everything else |
| **Supervised deep learning** (Modules 16–20) | many labelled examples per class | the labelled classes, fast | needs labels; cannot name a class it never saw (Module 19's open-set problem) |
| **Template-free** (this module, Module 22) | only the tomograms | recurring shapes, known *or unknown*, and their locations | patterns can be mixtures or artefacts; rare complexes are hard; needs careful validation |

The template-free promise is the real goal of visual proteomics: an **unbiased inventory** of a cell. Its price is that
the computer must decide by itself what counts as "the same shape", and how many shapes there are.

### 📄 Paper connection — Wu, Zeng, Zhu, Gao & Xu (2019), book chapter

- **Paper**: Wu X, Zeng X, Zhu Z, Gao X, Xu M. *Template-based and template-free approaches in cellular cryo-electron
  tomography structural pattern mining.* In: Husi H (ed.), *Computational Biology*, Chapter 11, pp. 175–186, Codon
  Publications (2019). [doi:10.15586/computationalbiology.2019.ch11](https://doi.org/10.15586/computationalbiology.2019.ch11) (open access)
- **Question it answers**: what are the main computational routes to "visual proteomics", and what are they used for?
- **What the chapter actually does**: reviews (1) template-based mining (template matching with statistical peak
  extraction, a knowledge-based scoring function, a GAN-based hypothesis test), (2) supervised classification and
  segmentation (2D-slice and 3D U-Net-type segmentation, SSN3D, DSRF3D/RB3D/CB3D classifiers, model compression,
  3D adversarial domain adaptation, multi-task learning) and (3) template-free mining (Multi-Pattern Pursuit; a
  convolutional autoencoder with k-means and weakly supervised segmentation), plus applications in bacterial cell
  biology and medicine. It contains no equations or new experiments.
- **What our notebook keeps**: its map of the field (the table above) and its template-free branch (Sections 2–3).
- **Result reported**: not applicable (review chapter).

---
## 2 · Template-free detection (Xu, Beck & Alber 2011)

The 2011 paper asked: given one tomogram and no templates, can we find **frequently recurring complexes**, with their
shapes and locations? Its pipeline has four steps, which we rebuild on a small simulated tomogram:

1. **Candidate objects** from local intensity maxima grown into regions by a watershed (not difference-of-Gaussians).
2. **Rotation-invariant features** for every voxel, describing its neighbourhood in a way that does not depend on how
   the complex is turned.
3. **Object similarity and clustering**: objects that contain many voxels with similar features are similar.
4. **Refinement by a Markov random field** whose neighbourhood joins spatial neighbours *and* feature-space neighbours.

### 2.1 A small in silico tomogram with an answer key

In [ ]:
# ── 2.1 Build a crowded volume with three complexes, and its voxel-level answer key ──
t0 = time.time()
DET_CLASSES = ["4v4r", "3j9i", "1bxn"]                       # ribosome, 20S proteasome, RuBisCO
templates = load_zoo(DET_CLASSES, size=32, sigma=1.0)
SHAPE = (48, 120, 120)                                       # z, y, x in voxels of 1 nm
clean, particles = make_crowded_cell(templates, {"4v4r": 7, "3j9i": 10, "1bxn": 12}, shape=SHAPE,
                                     rng=np.random.default_rng(0), membrane=False)


def label_volume(templates, particles, shape, level=0.3):
    """Answer key: each voxel gets the class (1, 2, …) of the particle whose density there is highest above level·max."""
    lab = np.zeros(shape, np.int16)
    best = np.zeros(shape, np.float32)
    index = {c: i + 1 for i, c in enumerate(templates)}
    for p in particles:
        t = templates[p["cls"]]
        sub = rotate_volume(t, p["R"])
        lo = np.round(p["center_zyx"]).astype(int) - np.array(sub.shape) // 2
        sl_v, sl_s = [], []
        for a in range(3):
            a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
            sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
        piece, region, bregion = sub[tuple(sl_s)], lab[tuple(sl_v)], best[tuple(sl_v)]
        upd = (piece > level * t.max()) & (piece > bregion)
        region[upd] = index[p["cls"]]; bregion[upd] = piece[upd]
    return lab


truth_vox = label_volume(templates, particles, SHAPE)
tomo = simulate_tomogram(clean, np.random.default_rng(1), snr=1.0, tilt_max_deg=60, ctf_pixel_A=None)
print({ZOO[c]["name"]: sum(p["cls"] == c for p in particles) for c in DET_CLASSES},
      f"placed in a {SHAPE[2]}×{SHAPE[1]}×{SHAPE[0]} nm box; SNR 1, ±60° wedge ({time.time() - t0:.1f} s)")

We switch off the CTF here (Module 06) to keep the toy simple; the paper's simulations included CTF, MTF and noise at
several SNRs (the range given in its text is about 0.5 to 5, plus noise-free).

### 2.2 Step 1: candidate objects from local maxima and a watershed

Smooth the tomogram with a Gaussian (only for this step), find all **local maxima** (voxels brighter than every
neighbour within a few voxels), keep the bright ones as seeds, and let each seed **grow** downhill over the smoothed
density, like water filling basins in an upside-down landscape: that is the **watershed** transform. Each basin becomes
a candidate object $S_s$.

In [ ]:
# ── 2.2 Candidates: local maxima → seeded watershed ──
def detect_candidates(vol, sigma=2.0, peak_size=7, k_hi=1.5, k_lo=0.5):
    """Template-free candidate objects: Gaussian-smoothed local maxima (> mean + k_hi·sd) grown by a seeded watershed
    inside the mask (> mean + k_lo·sd). Returns (label volume of objects, seed coordinates)."""
    sm = ndimage.gaussian_filter(vol, sigma)
    peaks = np.argwhere((sm == ndimage.maximum_filter(sm, size=peak_size)) & (sm > sm.mean() + k_hi * sm.std()))
    markers = np.zeros(vol.shape, np.int32)
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    return watershed(-sm, markers, mask=sm > sm.mean() + k_lo * sm.std()), peaks


t0 = time.time()
objects, seeds = detect_candidates(tomo)
n_obj = objects.max()
centres = np.array([np.round(p["center_zyx"]).astype(int) for p in particles])
found = np.mean([objects[tuple(c)] > 0 for c in centres])
print(f"{len(particles)} true particles → {n_obj} candidate objects; {100 * found:.0f} % of true centres lie inside an "
      f"object ({time.time() - t0:.1f} s)")
# the true class of each object = the most common non-background class of the answer key inside it
obj_truth = np.array([np.bincount(truth_vox[objects == k + 1], minlength=4)[1:].argmax() + 1
                      if (truth_vox[objects == k + 1] > 0).any() else 0 for k in range(n_obj)])

z0 = SHAPE[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
axes[0].imshow(tomo[z0]); axes[0].set_title(f"tomogram slice z = {z0} nm (SNR 1)")
axes[1].imshow(truth_vox[z0], cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
axes[1].set_title("answer key (1 ribosome, 2 proteasome, 3 RuBisCO)", fontsize=8)
axes[2].imshow(np.where(objects[z0] > 0, objects[z0] % 20 + 1, 0), cmap="tab20", interpolation="nearest")
near = seeds[np.abs(seeds[:, 0] - z0) <= 3]
axes[2].scatter(near[:, 2], near[:, 1], c="w", s=8)
axes[2].set_title("candidate objects (colours) and seeds (dots)", fontsize=8)
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With no template, simple image analysis found an object at essentially every particle. Some
particles are split into two basins (two maxima inside one ribosome), and objects are larger than the molecules because
smoothing spreads density. These imperfections are realistic; later steps must tolerate them.

### 2.3 Step 2: rotation-invariant voxel features

Recall Module 02 (and Xu, Zhang & Alber 2009). Around voxel $\mathbf x_i$, cut the neighbourhood into $M$ concentric
spherical **shells** $V_j(\mathbf x_i)=\{\mathbf x_k: r_{j-1}<|\mathbf x_k-\mathbf x_i|\le r_j\}$. On each shell the
density is a function on a sphere, which can be written as a sum of **spherical harmonics** (the 3D analogue of sines and
cosines on a circle):

$$g_j(\theta,\phi)\approx\sum_{l}\sum_{m=-l}^{l}a^{(j)}_{lm}\,Y_l^m(\theta,\phi),\qquad s_{jl}=\Big(\sum_{m=-l}^{l}\big|a^{(j)}_{lm}\big|^2\Big)^{1/2}.$$

*In words:* degree $l=0$ is the shell's average density, $l=1$ how lopsided it is, $l=2$ how elongated or flattened. A
rotation mixes the coefficients *within* each degree $l$ but never changes their total energy $s_{jl}$, so the numbers
$s_{jl}$ describe the neighbourhood **regardless of orientation**. The paper's extracted text names the per-shell
invariants but not their exact formula; the energy per degree above is the standard choice.

🧮 **Fast computation.** $a^{(j)}_{lm}(\mathbf x_i)$ is a correlation of the tomogram with the kernel "$Y_l^m$ restricted to
shell $j$", so one FFT-based correlation per (shell, $l$, $m$) gives it at **every voxel at once**. We use 4 shells
(radii up to 8.5 nm) and degrees $l=0,1,2$ (9 real harmonics), i.e. 36 correlations and 12 invariant numbers per voxel.

In [ ]:
# ── 2.3 Real spherical harmonics (l ≤ 2) and shell kernels ──
def real_sph_harm_l2(x, y, z):
    """Orthonormal real spherical harmonics of degree 0, 1, 2 at unit vectors (x, y, z). Returns {l: [arrays]}."""
    c1, c2 = np.sqrt(3 / (4 * np.pi)), 0.5 * np.sqrt(15 / np.pi)
    return {0: [0.5 * np.sqrt(1 / np.pi) * np.ones_like(x)],
            1: [c1 * y, c1 * z, c1 * x],
            2: [c2 * x * y, c2 * y * z, 0.25 * np.sqrt(5 / np.pi) * (3 * z * z - 1), c2 * x * z,
                0.25 * np.sqrt(15 / np.pi) * (x * x - y * y)]}


SHELLS = [(0.5, 2.5), (2.5, 4.5), (4.5, 6.5), (6.5, 8.5)]       # shell radii in voxels (= nm)


def shell_kernels(shells=SHELLS):
    """For each (shell j, degree l): the list of kernels Y_lm restricted to the shell, normalised by the shell size."""
    rad = int(np.ceil(shells[-1][1]))
    g = np.arange(-rad, rad + 1)
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r = np.sqrt(x**2 + y**2 + z**2) + 1e-12
    Y = real_sph_harm_l2(x / r, y / r, z / r)
    out = []
    for r0, r1 in shells:
        inside = (r > r0) & (r <= r1)
        for l in range(3):
            out.append([np.where(inside, Ylm, 0) / inside.sum() for Ylm in Y[l]])
    return out


def invariant_features(vol, shells=SHELLS):
    """s_jl = √(Σ_m a_jlm²) at every voxel (FFT correlations). Returns array (z, y, x, n_shells·3)."""
    F = np.fft.rfftn(vol)
    feats = []
    for kernels in shell_kernels(shells):
        energy = 0
        for k in kernels:
            pad = np.zeros(vol.shape)
            h = k.shape[0] // 2
            pad[np.ix_(*[np.arange(-h, h + 1) % n for n in vol.shape])] = k     # kernel centred on voxel (0,0,0)
            a = np.fft.irfftn(F * np.conj(np.fft.rfftn(pad)), s=vol.shape, axes=(0, 1, 2))
            energy = energy + a * a
        feats.append(np.sqrt(energy))
    return np.stack(feats, -1).astype(np.float32)


# invariance check on a clean ribosome: features at the centre for three random orientations
T = templates["4v4r"]
f_ref = invariant_features(T)[16, 16, 16]
for trial in range(3):
    R = random_rotation(rng)
    f_rot = invariant_features(rotate_volume(T, R))[16, 16, 16]
    raw = pearson(T[13:20, 13:20, 13:20], rotate_volume(T, R)[13:20, 13:20, 13:20])
    print(f"rotation {trial}: relative feature change {np.linalg.norm(f_rot - f_ref) / np.linalg.norm(f_ref):.3f}"
          f"   vs   correlation of the raw 7³ neighbourhood {raw:.2f}")

🔍 **What just happened?** After a random rotation, the 12 invariant numbers at the ribosome's centre change by only a
few percent (interpolation and the half-voxel offset of the rotation centre), whereas the raw voxel values of the same
neighbourhood are scrambled (low correlation). Invariant features let us compare neighbourhoods of differently oriented
copies **without aligning them**, which is exactly what a template-free method needs.

In [ ]:
# ── 2.3 Features for the whole tomogram (on the unfiltered intensities, as in the paper) ──
t0 = time.time()
feat = invariant_features(tomo)
fg = objects > 0
Fz = (feat[fg] - feat[fg].mean(0)) / feat[fg].std(0)          # standardised features of voxels inside objects
print(f"features: {feat.shape} in {time.time() - t0:.1f} s; {fg.sum()} voxels inside candidate objects")

### 2.4 Step 3: comparing and clustering objects

Two copies of the same complex, in any orientations, contain the same *kinds* of voxel neighbourhoods (the same mix of
"core", "edge", "groove" feature vectors). The paper therefore (i) groups voxels with similar feature vectors (it used
BIRCH clustering; we use k-means, giving each voxel a "feature word"), (ii) defines the dissimilarity of objects $a,b$ as

$$o_{a,b} = 1 - (\text{fraction of voxels with similar features in both objects}),$$

which we compute as one minus the overlap of the two objects' word histograms, $1-\sum_w\min(h_a(w),h_b(w))$, and
(iii) clusters objects hierarchically on $o_{a,b}$, discarding clusters with fewer than 3 objects. The paper cut the tree
at the minimum of the Kelley *et al.* (1996) penalty; we reuse the silhouette criterion of Module 14.

In [ ]:
# ── 2.4 Voxel feature words → object histograms → hierarchical clustering ──
t0 = time.time()
N_WORDS = 12
words_model = MiniBatchKMeans(N_WORDS, random_state=0, n_init=3, batch_size=4096).fit(Fz[::3])
words = words_model.predict(Fz)                                   # one word per voxel inside an object
H = np.zeros((n_obj, N_WORDS))
np.add.at(H, (objects[fg] - 1, words), 1)
H /= H.sum(1, keepdims=True)
O = 1 - np.minimum(H[:, None, :], H[None, :, :]).sum(-1)          # o_ab for every pair of objects
np.fill_diagonal(O, 0)
Zobj = linkage(squareform(O, checks=False), method="average")
sil = {k: silhouette_score(O, fcluster(Zobj, k, "maxclust") - 1, metric="precomputed") for k in range(2, 8)}
K_obj = max(sil, key=sil.get)
obj_class = fcluster(Zobj, K_obj, "maxclust") - 1
sizes = np.bincount(obj_class)
obj_class = np.where(sizes[obj_class] >= 3, obj_class, -1)       # clusters of < 3 objects are discarded
print("silhouette per K:", {k: round(v, 3) for k, v in sil.items()})
print("ARI vs truth per K:", {k: round(adjusted_rand_score(obj_truth, fcluster(Zobj, k, "maxclust")), 2) for k in range(2, 8)})
print(f"chosen K = {K_obj}; ARI of object classes vs truth = {adjusted_rand_score(obj_truth, obj_class):.2f} "
      f"({time.time() - t0:.1f} s)")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
order = np.argsort(obj_class * 100 + obj_truth)
im = axes[0].imshow(O[np.ix_(order, order)], cmap="viridis")
axes[0].set_title("object dissimilarity $o_{a,b}$ (sorted by cluster)", fontsize=9)
plt.colorbar(im, ax=axes[0], fraction=0.046)
ct = np.zeros((K_obj, 3), int)
for c, t in zip(obj_class, obj_truth):
    if c >= 0:
        ct[c, t - 1] += 1
bottom = np.zeros(K_obj)
for t in range(3):
    axes[1].bar(np.arange(K_obj), ct[:, t], bottom=bottom, label=ZOO[DET_CLASSES[t]]["name"]); bottom += ct[:, t]
axes[1].set_xlabel("discovered object cluster"); axes[1].set_ylabel("number of objects")
axes[1].set_title("what each cluster contains (answer key)", fontsize=9); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without any template, objects were grouped by the *kinds* of neighbourhoods they contain.
Look at the bars: the cleanest split is by **size** — the small RuBisCO objects form a cluster of their own, while
ribosomes and proteasomes end up together. The silhouette prefers this two-cluster cut, and even with more clusters the
two large complexes are not separated cleanly (the ARI per K stays at or below about 0.5). Our features are coarse — 12
rotation-invariant numbers per voxel (degrees l ≤ 2 on 4 shells) — and two complexes of similar size and density look
alike to them; the paper used richer features (✏️ Bonus: add l = 3). Still, with no template at all we have learned
that the tomogram contains at least two recurring kinds of object, how many copies of each, and where they are.

### 2.5 Step 4: a Markov random field that lets distant copies talk to each other

Object-level clustering can still make mistakes, especially for small or split objects. The paper refines the labels
voxel by voxel with a **Gaussian hidden Markov random field (GHMRF)**:

- Each voxel $i$ has a hidden label $l_i$ (a pattern class or background) and an observation $f_i$ with a Gaussian
  emission $f_i\mid l_i\sim\mathcal N(\mu_{l},\sigma_{l}^2)$.
- A **Potts prior** rewards neighbours that share a label: clique potential $V_c(l_i,l_j)=-\delta_{l_i l_j}$ (−1 if equal,
  0 otherwise).
- The key idea: the neighbourhood of a voxel is its **grid neighbours plus its feature-space neighbours** (the 20 most
  similar voxels at least $r_d$ away in the tomogram). So a voxel in one ribosome is "connected" to similar voxels in
  *other* ribosomes far away, and copies of a complex share label information.
- Find labels minimising $U(\mathbf f\mid\mathbf l)+U(\mathbf l)$ by **iterated conditional modes (ICM)**: give each voxel
  the label with the lowest energy given its neighbours, re-estimate $\mu_l,\sigma_l$ (an EM-like step), repeat. Finally
  each object takes the **majority label** of its voxels.

In energy form, for voxel $i$ and candidate label $l$:

$$E_i(l)=\underbrace{\tfrac12\sum_d\Big[\frac{(f_{id}-\mu_{ld})^2}{\sigma_{ld}^2}+\log\sigma_{ld}^2\Big]}_{\text{how well the features fit class }l}\;-\;\beta\!\!\sum_{j\in\mathcal N^{grid}_i\cup\mathcal N^{fea}_i}\!\!\delta_{l,l_j}.$$

*In words:* a voxel prefers the class whose typical features it has, and the class most of its spatial *and* feature-space
neighbours have. Our toy uses the 12-dimensional feature vector with a diagonal Gaussian per class, restricts the field to
voxels inside detected objects (no background class) and uses 6 feature-space neighbours found among a random sample of
voxels, to stay fast.

**A controlled experiment.** To see what the feature-space neighbours add, we deliberately **corrupt** the object labels
of 7 objects (as if the clustering had made mistakes) and ask whether the MRF can repair them, with grid neighbours only
versus grid + feature-space neighbours.

In [ ]:
# ── 2.5 Neighbourhoods for the MRF: 6 grid neighbours + feature-space neighbours ──
t0 = time.time()
vox = np.argwhere(fg)                                             # coordinates of the field's voxels
n_vox = len(vox)
index_of = -np.ones(SHAPE, int); index_of[fg] = np.arange(n_vox)
grid_nb = []
for d in [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]:
    q = vox + np.array(d)
    ok = np.all((q >= 0) & (q < np.array(SHAPE)), axis=1)
    nb = np.full(n_vox, -1); nb[ok] = index_of[tuple(q[ok].T)]
    grid_nb.append(nb)
grid_nb = np.stack(grid_nb, 1)

P6 = PCA(6, random_state=0).fit_transform(Fz)                    # compact features for the neighbour search
anchors = np.random.default_rng(0).choice(n_vox, 6000, replace=False)
cand = anchors[NearestNeighbors(n_neighbors=12).fit(P6[anchors]).kneighbors(P6, return_distance=False)]
far = np.linalg.norm(vox[cand] - vox[:, None, :], axis=-1) >= 8   # feature neighbours must be ≥ r_d = 8 nm away
first = np.argsort(~far, axis=1, kind="stable")[:, :6]
feat_nb = np.take_along_axis(np.where(far, cand, -1), first, 1)
print(f"{n_vox} voxels; grid and feature-space neighbourhoods built in {time.time() - t0:.1f} s")


def icm_potts(labels, neighbours, beta, n_sweeps=6, seed=0):
    """Iterated conditional modes for a Potts MRF with diagonal-Gaussian emissions on the features Fz."""
    lab, r = labels.copy(), np.random.default_rng(seed)
    K = lab.max() + 1
    for _ in range(n_sweeps):
        mus = np.array([Fz[lab == k].mean(0) for k in range(K)])                 # EM-like re-estimation
        var = np.array([Fz[lab == k].var(0) + 1e-3 for k in range(K)])
        E = 0.5 * (((Fz[:, None, :] - mus[None])**2 / var[None]) + np.log(var[None])).sum(-1)
        for _half in range(2):                                    # update a random half at a time (avoids oscillation)
            agree = np.zeros((n_vox, K))
            for col in range(neighbours.shape[1]):
                nb = neighbours[:, col]; ok = nb >= 0
                np.add.at(agree, (np.where(ok)[0], lab[nb[ok]]), 1)
            sel = r.random(n_vox) < 0.5
            lab[sel] = (E - beta * agree).argmin(1)[sel]
    return lab


def object_majority(voxel_labels, K):
    """Each object takes the most common label of its voxels."""
    votes = np.zeros((n_obj, K)); np.add.at(votes, (objects[fg] - 1, voxel_labels), 1)
    return votes.argmax(1)

In [ ]:
# ── 2.5 Corrupt 7 object labels, then let the MRF repair them ──
t0 = time.time()
good = np.where(obj_class >= 0, obj_class, 0)                     # the (clean) clustering result of 2.4
K_mrf = good.max() + 1
bad = good.copy()
flip = np.random.default_rng(3).choice(n_obj, 7, replace=False)
bad[flip] = (good[flip] + np.random.default_rng(4).integers(1, K_mrf, len(flip))) % K_mrf
start = bad[objects[fg] - 1]                                      # voxel labels inherited from (corrupted) objects
repair = {}
for name, nbs in [("grid only", grid_nb), ("grid + feature space", np.concatenate([grid_nb, feat_nb], 1))]:
    voxel_lab = icm_potts(start, nbs, beta=3.0)
    repair[name] = (object_majority(voxel_lab, K_mrf), voxel_lab)
    print(f"{name:22s}: objects still wrong {np.sum(repair[name][0] != good)} of {len(flip)} corrupted "
          f"(voxels relabelled: {100 * np.mean(voxel_lab != start):.0f} %)")
print(f"({time.time() - t0:.1f} s)")

In [ ]:
# ── 2.5 Figure: what the two fields did ──
def paint(obj_labels):
    """Volume where every object voxel shows its object's label (+1; 0 = outside objects)."""
    v = np.zeros(SHAPE, int); v[fg] = obj_labels[objects[fg] - 1] + 1
    return v


fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, (title, vol) in zip(axes, [("clean clustering (2.4)", paint(good)), ("after corrupting 7 objects", paint(bad)),
                                    ("MRF, grid neighbours only", paint(repair["grid only"][0])),
                                    ("MRF, grid + feature space", paint(repair["grid + feature space"][0]))]):
    ax.imshow(vol[z0], cmap="tab10", vmin=0, vmax=9, interpolation="nearest"); ax.set_title(title, fontsize=8); ax.axis("off")
plt.suptitle(f"object labels on slice z = {z0} nm (colours = discovered classes)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With only grid neighbours, the voxels of a mislabelled object mostly agree *with each other*,
so the wrong label is self-sustaining: the MRF can only polish boundaries. With feature-space neighbours, each voxel also
"listens" to similar voxels inside *other* objects, most of which are correctly labelled, and several corrupted objects
flip back to the right class after the majority vote. That is the paper's insight in one experiment: **copies of the
same complex anywhere in the cell can vote on each other's identity**.

✏️ **Your turn.** (a) Change `beta=3.0` to `beta=0.5` or `beta=10.0`: what happens when the prior is too weak or too
strong? (b) In 2.1 lower the SNR to 0.3: which step breaks first, detection (2.2) or clustering (2.4)?

### 📄 Paper connection — Xu, Beck & Alber (2011)

- **Paper**: Xu M, Beck M, Alber F. *Template-free detection of macromolecular complexes in cryo electron tomograms.*
  Bioinformatics 27(13):i69–i76 (ISMB 2011). [doi:10.1093/bioinformatics/btr207](https://doi.org/10.1093/bioinformatics/btr207)
- **Question it answers**: can frequently recurring complexes, with their shapes and locations, be discovered in a
  tomogram without any template?
- **What the paper actually does**: candidate objects from local intensity maxima seeding a modified watershed (Gaussian
  smoothing used only for segmentation; classification on unfiltered intensities); per-voxel rotation-invariant
  features from spherical-harmonic expansions on concentric shells; voxel clustering (BIRCH) and feature-space
  neighbours (20 most similar voxels at least $r_d$ away); object dissimilarity $o_{a,b}$; hierarchical clustering of
  objects cut at the minimum of the Kelley penalty (clusters < 3 objects dropped); refinement by a Gaussian hidden MRF
  with a Potts prior over grid + feature-space neighbours, optimised by ICM alternating with EM; majority label per object.
- **What our toy keeps**: every step above in simplified form.
- **What our toy leaves out**: BIRCH (we use k-means words), the exact invariant formula, the Kelley penalty (we use the
  silhouette), a background class in the MRF, realistic CTF/MTF simulation and the paper's 52 benchmark sets.
- **Result reported**: ⚠️ the research notes for this course could read the paper's methods and data but not its
  quantitative results; the abstract describes the work as a proof of principle on realistic simulations. The simulated
  data: 24 complexes in 52 sets of 4, 10 copies each, in 150 × 150 × 100 nm volumes. Read the paper for its numbers.
- **Code**: none in AITom (AITom's `aitom/pick/dog` is the later difference-of-Gaussians picker, not this method).

---
## 3 · De novo structural pattern mining: Multi-Pattern Pursuit (Xu *et al.* 2019)

Template-free *detection* finds where things are. **Multi-Pattern Pursuit (MPP)** goes further: from thousands of
subtomograms, find **all** abundant recurring structures ("patterns"), each with an average and a member list, without
knowing how many there are and without assuming the classes are well separated.

### 3.1 How good is a pattern? From SSNR to SFSC

Take the members of a candidate pattern, aligned. For every Fourier coefficient $\xi$ compute the mean $\mu(\xi)$ and the
variance $\sigma^2(\xi)$ over the members that observed it ($\tilde M(\xi)$ of them, because of the wedges). The
**spectral signal-to-noise ratio** of the average in a frequency shell of radius $r$ is

$$\eta_r=\frac{\sum_{||\xi|-r|\le\Delta r}\tilde M(\xi)\,|\mu(\xi)|^2}{\sum_{||\xi|-r|\le\Delta r}\sigma^2(\xi)} .$$

*In words:* how strong the consistent part (the mean) is compared with the scatter between members, with each coefficient
counted as often as it was measured. Then MPP converts it to an **estimated FSC** and sums over shells:

$$\rho_r=\frac{\eta_r}{2+\eta_r},\qquad \tilde\rho=\sum_r\rho_r\quad(\text{SFSC, the "sum of FSC"}).$$

🧮 **Why $\eta/(2+\eta)$?** If you split the members into two halves, each half-average has half the SSNR, $\eta/2$.
The expected FSC between two independent half-averages with SSNR $s$ each is $s/(1+s)$ (Module 10). Substituting
$s=\eta/2$ gives $\rho=\eta/(2+\eta)$: MPP predicts the gold-standard FSC of a pattern *without* splitting it. A high SFSC
means "large, homogeneous and well aligned". (Note: for pure noise this $\eta$ estimate is about 1, not 0, so every
shell contributes a floor of about 1/3; some SSNR estimators subtract 1 to remove it. Rankings between patterns are
unaffected, but SFSC values are not on an absolute scale.)

### 3.2 The MPP loop

0. Pick particles (DoG) and optionally **pose-normalise** them (centre of mass + principal axes).
1. **Pattern generation**: fill each aligned particle's missing wedge with the corresponding coefficients of the pattern
   it is aligned to (*imputation*), keep informative voxels (high average neighbour covariance, as in Module 14), reduce
   dimension (PCA) and run **k-means with more clusters than expected** (over-clustering).
2. Score every candidate average by **SFSC** and add it to a growing **library** of candidate patterns.
3. **Greedy disjoint selection**: walk down the library by SFSC, accepting a pattern only if its members overlap little
   with those already accepted.
4. Align the selected averages to a common frame, then align **every** particle to **every** selected pattern.
5. **Redundancy removal**: for patterns A and B, where B has the higher SFSC, compare the alignment scores of A's members
   against A and against B with a **Wilcoxon signed-rank test** (significance 0.01). If A's members do not significantly
   prefer A, A is redundant and dropped.
6. Each particle keeps its best-scoring transform; repeat until the selected patterns stop changing.

We run this loop in flatland, reusing the 2D aligner of Module 14.

In [ ]:
# ── 3.2 Flatland tools from Module 14 (wedges, transforms, fast rotational + translational search) ──
S2, N_THETA, PAD, TILT = 40, 180, 2, 60.0
ANG = 2 * np.pi * np.arange(N_THETA) / N_THETA
KY, KX = freq_grid((S2, S2))
SHIFTS = np.fft.fftfreq(S2) * S2
N2 = S2 * PAD
RINGS = np.arange(4, 25).astype(float)
_yy, _xx = np.indices((S2, S2))
APOD = np.clip((S2 / 2 - 1 - np.sqrt((_yy - (S2 - 1) / 2)**2 + (_xx - (S2 - 1) / 2)**2)) / 3.0, 0, 1)


def wedge2d(shape, theta=0.0, tilt_max_deg=TILT):
    """2D missing-wedge mask seen from a frame rotated by theta: W(R_θᵀ ξ)."""
    ky, kx = freq_grid(shape)
    c, s = np.cos(theta), np.sin(theta)
    m = np.abs(-s * kx + c * ky) <= np.abs(c * kx + s * ky) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[0, 0] = True
    return m.astype(np.float32)


W0 = wedge2d((S2, S2))


def transform_image(img, theta, t):
    """T_(θ,t): rotate by θ about the centre, then shift by t = (tx, ty)."""
    return rotate_image(img, theta, shift_xy=t)


def flatland_particles(images, names, snr, rng, max_shift=2.0):
    """2D 'subtomograms': random rotation + shift, missing wedge, noise confined to the wedge."""
    X = []
    for name in names:
        clean_img = apply_fourier_mask(transform_image(images[name], rng.uniform(0, 2 * np.pi),
                                                       rng.uniform(-max_shift, max_shift, 2)), W0)
        X.append(apply_fourier_mask(add_noise_snr(clean_img, snr, rng), W0))
    return np.array(X, np.float32)


def _polar_matrix():
    """Sparse bilinear map: padded, centred spectrum → (ring, angle) samples."""
    rr, pp = np.meshgrid(RINGS, ANG, indexing="ij")
    y, x = N2 // 2 + rr * np.sin(pp), N2 // 2 + rr * np.cos(pp)
    y0, x0 = np.floor(y).astype(int), np.floor(x).astype(int)
    fy, fx = y - y0, x - x0
    cols = np.stack([y0 * N2 + x0, y0 * N2 + x0 + 1, (y0 + 1) * N2 + x0, (y0 + 1) * N2 + x0 + 1], -1).ravel()
    vals = np.stack([(1 - fy) * (1 - fx), (1 - fy) * fx, fy * (1 - fx), fy * fx], -1).ravel()
    return sparse.csr_matrix((vals, (np.repeat(np.arange(y.size), 4), cols)), shape=(y.size, N2 * N2))


POLAR = _polar_matrix()


def polar_spectra(imgs, wedge_angle=0.0):
    """Polar FFTs of |F|·M, |F|²·M and M for a batch (M = None → full coverage): the inputs of c' (Xu 2012)."""
    imgs = np.asarray(imgs, float) * APOD
    o = (N2 - S2) // 2
    pad = np.zeros((len(imgs), N2, N2)); pad[:, o:o + S2, o:o + S2] = imgs
    A = np.abs(np.fft.fftshift(np.fft.fft2(pad), axes=(1, 2))); A[:, N2 // 2, N2 // 2] = 0
    m = np.ones((N2, N2)) if wedge_angle is None else np.fft.fftshift(wedge2d((N2, N2), wedge_angle))

    def to_polar(a):
        return (POLAR @ a.reshape(len(a), -1).T).T.reshape(len(a), len(RINGS), N_THETA)
    return (np.fft.fft(to_polar(A * m), axis=-1), np.fft.fft(to_polar(A**2 * m), axis=-1),
            np.fft.fft(to_polar(m[None]), axis=-1)[0])


def rot_scores(a, b):
    """c'(θ) = c0/√(c1·c2) for all angles at once (1D FFTs along the polar angle)."""
    w = RINGS[:, None]
    c0 = (np.fft.ifft(a[0] * np.conj(b[0]), axis=-1).real * w).sum(-2)
    c1 = (np.fft.ifft(a[1] * np.conj(b[2]), axis=-1).real * w).sum(-2)
    c2 = (np.fft.ifft(a[2] * np.conj(b[1]), axis=-1).real * w).sum(-2)
    return c0 / np.sqrt(np.maximum(c1 * c2, 1e-12))


def top_peaks(scores, n):
    """The n highest circular local maxima along the last axis."""
    is_peak = (scores >= np.roll(scores, 1, -1)) & (scores >= np.roll(scores, -1, -1))
    return np.argsort(-np.where(is_peak, scores, -np.inf), axis=-1)[..., :n]


def translation_search(FX, G, Om, max_shift):
    """Constrained correlation over all shifts; returns (best score, (tx, ty))."""
    num = np.fft.ifft2(FX * Om * np.conj(G * Om)).real
    den = np.sqrt((np.abs(FX)**2 * Om).sum((-2, -1)) * (np.abs(G)**2 * Om).sum((-2, -1))) / (S2 * S2)
    cc = num / np.maximum(den, 1e-12)[..., None, None]
    allowed = (np.abs(SHIFTS)[:, None] <= max_shift) & (np.abs(SHIFTS)[None, :] <= max_shift)
    cc = np.where(allowed, cc, -np.inf).reshape(cc.shape[:-2] + (-1,))
    k = cc.argmax(-1)
    return np.take_along_axis(cc, k[..., None], -1)[..., 0], np.stack([SHIFTS[k % S2], SHIFTS[k // S2]], -1)


def align_to_reference(FX, PX, ref, n_cand=4, max_shift=3):
    """Every particle (wedge W0) onto a full reference: X_i ≈ T_(θ,t)(ref); best candidate first."""
    bank = np.fft.fft2(np.array([rotate_image(ref, a) for a in ANG])); bank[:, 0, 0] = 0
    js = top_peaks(rot_scores(PX, polar_spectra(ref[None], None)), n_cand)
    FXd = FX.copy(); FXd[:, 0, 0] = 0
    best, t = translation_search(FXd[:, None], bank[js], W0, max_shift)
    order = np.argsort(-best, axis=1)
    return (np.take_along_axis(best, order, 1), np.take_along_axis(js, order, 1),
            np.take_along_axis(t, order[..., None], 1))


def align_full(ref, img, n_cand=4, max_shift=4):
    """Full-coverage image onto another: ref ≈ T_(θ,t)(img). Returns (score, θ, t)."""
    js = top_peaks(rot_scores(polar_spectra(ref[None], None), polar_spectra(img[None], None))[0], n_cand)
    G = np.fft.fft2(np.array([rotate_image(img, ANG[j]) for j in js])); G[:, 0, 0] = 0
    F = np.fft.fft2(ref); F[0, 0] = 0
    best, t = translation_search(F[None], G, np.ones((S2, S2)), max_shift)
    k = best.argmax()
    return best[k], ANG[js[k]], t[k]


def compose_to_frame(phi, psi):
    """x ≈ T_phi(template), template_0 ≈ T_psi(template)  →  (α, u) with T_(α,u)(x) in template_0's frame."""
    (th, t), (thp, s) = phi, psi
    a = thp - th
    c, sn = np.cos(a), np.sin(a)
    return a, np.asarray(s, float) - np.array([[c, -sn], [sn, c]]) @ np.asarray(t, float)


def to_common(X, alphas, us):
    """Particles moved into the common frame, with their rotated wedge masks."""
    Xc = np.array([transform_image(x, a, u) for x, a, u in zip(X, alphas, us)])
    return Xc, np.array([wedge2d((S2, S2), a) for a in alphas])


def wedge_average(Xc, Mc):
    """Missing-wedge-weighted average of particles in a common frame (Module 14)."""
    F = (np.fft.fft2(Xc) * Mc).sum(0); n = Mc.sum(0)
    return np.real(np.fft.ifft2(np.where(n > 0, F / np.maximum(n, 1), 0)))


def feature_weights(Xc):
    """Average covariance of each pixel with its 8 neighbours across particles (≥ 0, max 1)."""
    D = Xc - Xc.mean(0)
    cov = np.mean([(D * np.roll(D, (dy, dx), axis=(1, 2))).mean(0)
                   for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)], axis=0)
    W = np.clip(cov, 0, None)
    return W / (W.max() + 1e-12)


print("flatland tools ready")

In [ ]:
# ── 3.1 SSNR → ρ = η/(2+η) → SFSC ──
RING_INDEX = np.minimum(np.round(np.sqrt(KX**2 + KY**2) * S2).astype(int), S2 // 2)


def ssnr_sfsc(Xc, Mc, r_max=S2 // 2 - 1):
    """MPP's pattern score for aligned particles Xc with wedge masks Mc. Returns (SFSC, ρ per shell, η per shell)."""
    F = np.fft.fft2(Xc) * Mc
    n = Mc.sum(0)                                                   # M̃(ξ): how many members observed ξ
    mu = np.where(n > 0, F.sum(0) / np.maximum(n, 1), 0)
    ok = n >= 2                                                     # a variance needs at least two observations
    var = np.where(ok, ((np.abs(F)**2).sum(0) - n * np.abs(mu)**2) / np.maximum(n - 1, 1), 0)
    num = np.bincount(RING_INDEX[ok], (n * np.abs(mu)**2)[ok], S2 // 2 + 1)[1:r_max + 1]
    den = np.bincount(RING_INDEX[ok], var[ok], S2 // 2 + 1)[1:r_max + 1]
    eta = num / np.maximum(den, 1e-12)
    rho = eta / (2 + eta)
    return float(rho.sum()), rho, eta


print("SFSC ready")

### 3.3 What SFSC rewards (and why rare complexes are hard)

Before running MPP, let us look at the score itself. We build sets of perfectly aligned particles (true poses, so only
the scoring is tested): **pure** sets of each molecule, and **mixed** sets (half one molecule, half another), of
different sizes, and compute their SFSC.

In [ ]:
# ── 3.3 The molecules for Section 3, and SFSC of pure vs mixed sets ──
MPP_CLASSES = ["4v4r", "4v94", "1bxn", "1s3x"]
zoo40 = load_zoo(MPP_CLASSES, size=S2, sigma=1.0, verbose=False)
flat = {pid: normalize(zoo40[pid].sum(axis=1)) for pid in MPP_CLASSES}
fig, axes = plt.subplots(1, 4, figsize=(7.5, 2.1))
for ax, pid in zip(axes, MPP_CLASSES):
    ax.imshow(flat[pid]); ax.set_title(f"{pid.upper()}\n{ZOO[pid]['name']}", fontsize=8); ax.axis("off")
plt.suptitle("flatland molecules for Multi-Pattern Pursuit (40 nm boxes)", fontsize=9)
plt.tight_layout(); plt.show()


def aligned_set(names, snr, rng):
    """Perfectly aligned particles (pose known): wedge at a random angle per particle, noise inside the wedge."""
    Xc, Mc = [], []
    for name in names:
        a = rng.uniform(0, 2 * np.pi)
        M = wedge2d((S2, S2), a)
        Xc.append(apply_fourier_mask(add_noise_snr(apply_fourier_mask(flat[name], M), snr, rng), M)); Mc.append(M)
    return np.array(Xc), np.array(Mc)


t0 = time.time()
sizes_sf = [8, 15, 30, 60, 120]
curves = {}
for label, mix in [("pure ribosome", ["4v4r"]), ("pure RuBisCO", ["1bxn"]), ("pure Hsp70 domain", ["1s3x"]),
                   ("50 % ribosome + 50 % TRiC pair", ["4v4r", "4v94"])]:
    curves[label] = [ssnr_sfsc(*aligned_set([mix[i % len(mix)] for i in range(n)], 0.5, rng))[0] for n in sizes_sf]
    print(f"{label:32s}", " ".join(f"{v:5.2f}" for v in curves[label]))
print("(columns: set sizes", sizes_sf, f") ({time.time() - t0:.1f} s)")
fig, ax = plt.subplots(figsize=(5.2, 3.2))
for label, ys in curves.items():
    ax.semilogx(sizes_sf, ys, "o-" if "pure" in label else "s--", label=label)
ax.set_xlabel("number of aligned members"); ax.set_ylabel("SFSC = Σ ρ_r"); ax.legend(fontsize=7)
ax.set_title("SFSC rewards size and homogeneity (flatland SNR 0.5)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** SFSC grows with the number of members (more members → higher SSNR in every shell) and levels
off, because no shell's ρ can exceed 1. For small and medium sets, a mixture scores lower than a pure set of the same
size (the two molecules disagree, which inflates the variance); for large sets every candidate approaches the ceiling
and the differences shrink. Two consequences for MPP: greedy selection by SFSC prefers **big, pure** patterns; and a
**rare** complex, even perfectly pure, has a small SFSC and must compete with much larger patterns. The paper reports
that complexes with fewer than about 200–300 copies were often missed in its simulations.

⚠️ Comparing *different* molecules is trickier. Here every image has the same SNR (a variance ratio over the box), so
the small Hsp70 domain carries as much signal per image as the ribosome — and even scores higher. In a real tomogram the
noise is the same for every molecule, so a 43 kDa domain carries roughly 50× less signal than a 2.3 MDa ribosome and
scores far lower.

In [ ]:
# ── 3.4 The mini Multi-Pattern Pursuit ──
def pose_normalize(x):
    """MPP's optional step 0 in 2D: centre of mass + principal axis of the smoothed, thresholded particle → (α, u)."""
    sm = ndimage.gaussian_filter(x, 2.0)
    m = sm > sm.mean() + sm.std()
    yy, xx = np.nonzero(m); w = sm[m]
    cy, cx = np.average(yy, weights=w), np.average(xx, weights=w)
    evals, evecs = np.linalg.eigh(np.cov(np.stack([xx - cx, yy - cy]), aweights=w))
    a = -np.arctan2(evecs[1, -1], evecs[0, -1])                    # rotate the major axis onto x
    c, s_ = np.cos(a), np.sin(a)
    centre = (S2 - 1) / 2
    return a, -(np.array([[c, -s_], [s_, c]]) @ np.array([cx - centre, cy - centre]))


def wilcoxon_redundant(scores_own, scores_other, alpha=0.01):
    """MPP redundancy rule for pattern A vs a better-scoring pattern B, given A's members' alignment scores to A and B.

    A is redundant unless its members prefer A significantly (two-sided Wilcoxon signed-rank test, then direction)."""
    if np.all(scores_own > scores_other):
        return False, 0.0
    _, p = wilcoxon(scores_other, scores_own)
    return bool(p > alpha or np.median(scores_other) > np.median(scores_own)), p


def multi_pattern_pursuit(X, n_pass=6, k_gen=6, overlap_max=0.5, size_min=8, alpha=0.01, labels_true=None, seed=0):
    """A flatland mini-MPP (Xu et al. 2019): generate → score (SFSC) → select → align → remove redundancy → repeat."""
    N = len(X)
    FX, PX = np.fft.fft2(X), polar_spectra(X, 0.0)
    poses = [pose_normalize(x) for x in X]                        # step 0
    imputing_template, library, history = None, [], []
    for p in range(n_pass):
        # 1. pattern generation: common frame → imputation → informative pixels → PCA → k-means (over-clustering)
        Xc, Mc = to_common(X, *zip(*poses))
        Fc = np.fft.fft2(Xc)
        if imputing_template is not None:
            Fc = Mc * Fc + (1 - Mc) * np.fft.fft2(imputing_template)
        Xi = np.real(np.fft.ifft2(Fc))
        W = feature_weights(Xi)
        keep = W.ravel() >= np.quantile(W, 0.5)
        Z = PCA(20, random_state=0).fit_transform(Xi.reshape(N, -1)[:, keep])
        lab = KMeans(k_gen, n_init=4, random_state=seed + p).fit_predict(Z)
        # 2. score every candidate by SFSC and add it to the library
        for k in range(k_gen):
            mem = np.where(lab == k)[0]
            if len(mem) >= size_min:
                library.append(dict(id=len(library), born=p, members=set(mem.tolist()),
                                    sfsc=ssnr_sfsc(Xc[mem], Mc[mem])[0], avg=wedge_average(Xc[mem], Mc[mem])))
        # 3. greedy disjoint selection, best SFSC first
        sel, covered = [], set()
        for c in sorted(library, key=lambda c: -c["sfsc"]):
            if len(covered & c["members"]) / len(c["members"]) <= overlap_max:
                sel.append(c); covered |= c["members"]
        # 4. align the patterns to a common frame, then every particle to every pattern
        psi = [(0.0, np.zeros(2))] + [align_full(sel[0]["avg"], c["avg"])[1:] for c in sel[1:]]
        S, best_pose = np.zeros((N, len(sel))), []
        for k, c in enumerate(sel):
            s_, j_, t_ = align_to_reference(FX, PX, c["avg"])
            S[:, k] = s_[:, 0]; best_pose.append((ANG[j_[:, 0]], t_[:, 0]))
        # 5. redundancy removal: weakest pattern first, against every better pattern still alive
        alive = list(range(len(sel)))
        for a in sorted(range(len(sel)), key=lambda k: sel[k]["sfsc"]):
            mem = np.array(sorted(sel[a]["members"]))
            for b in [b for b in alive if b != a and sel[b]["sfsc"] > sel[a]["sfsc"]]:
                if wilcoxon_redundant(S[mem, a], S[mem, b], alpha)[0]:
                    alive.remove(a); break
        # 6. every particle keeps its best transform among the surviving patterns
        assign = np.array(alive)[S[:, alive].argmax(1)]
        poses = [compose_to_frame((best_pose[k][0][i], best_pose[k][1][i]), psi[k]) for i, k in enumerate(assign)]
        in_common = {k: transform_image(sel[k]["avg"], *psi[k]) for k in alive}
        imputing_template = np.array([in_common[k] for k in assign])
        rec = dict(pass_=p, library=len(library), selected=len(sel), redundant=len(sel) - len(alive),
                   ids=sorted(sel[k]["id"] for k in alive), assign=assign,
                   patterns=[(sel[k], np.where(assign == k)[0]) for k in alive])
        history.append(rec)
        comp = "" if labels_true is None else \
            str([np.bincount(labels_true[assign == k], minlength=labels_true.max() + 1).tolist() for k in alive])
        print(f"pass {p}: library {rec['library']:3d}, selected {rec['selected']}, redundant {rec['redundant']}, "
              f"kept {len(alive)}  | members by true class {comp}")
        if len(history) >= 3 and history[-1]["ids"] == history[-2]["ids"] == history[-3]["ids"]:
            print("selected patterns unchanged for 3 passes → stop")
            break
    return history


t0 = time.time()
COUNTS = [90, 50, 30, 15]                                          # unequal abundances; the Hsp70 domain is rare
labels_m = np.repeat(np.arange(4), COUNTS)
Xm = flatland_particles(flat, [MPP_CLASSES[l] for l in labels_m], snr=0.5, rng=np.random.default_rng(14))
mpp_hist = multi_pattern_pursuit(Xm, labels_true=labels_m)
final = mpp_hist[-1]
print(f"{len(final['patterns'])} patterns; ARI of pattern membership vs truth = "
      f"{adjusted_rand_score(labels_m, final['assign']):.2f}   ({time.time() - t0:.1f} s)")

In [ ]:
# ── 3.5 Figure: the patterns MPP discovered, and what they contain ──
pats = sorted(final["patterns"], key=lambda pm: -pm[0]["sfsc"])
fig, axes = plt.subplots(2, len(pats), figsize=(2.0 * len(pats), 4.2), squeeze=False)
for j, (pat, mem) in enumerate(pats):
    axes[0, j].imshow(pat["avg"]); axes[0, j].axis("off")
    axes[0, j].set_title(f"SFSC {pat['sfsc']:.1f}\n{len(mem)} members (born pass {pat['born']})", fontsize=7)
    comp = np.bincount(labels_m[mem], minlength=4)
    axes[1, j].bar(range(4), comp, color=["tab:red", "tab:blue", "tab:green", "tab:orange"])
    axes[1, j].set_xticks(range(4)); axes[1, j].set_xticklabels([c.upper() for c in MPP_CLASSES], fontsize=6, rotation=45)
    axes[1, j].set_ylim(0, max(COUNTS))
axes[1, 0].set_ylabel("members by true class")
plt.suptitle("Mini-MPP patterns (top) and their composition by the hidden answer key (bottom)", fontsize=9)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(5.2, 2.8))
passes = [h["pass_"] for h in mpp_hist]
ax.plot(passes, [h["selected"] for h in mpp_hist], "o-", label="selected (greedy, disjoint)")
ax.plot(passes, [h["selected"] - h["redundant"] for h in mpp_hist], "s-", label="kept after Wilcoxon test")
ax.axhline(4, color="k", ls=":", label="true number of molecules")
ax.set_xlabel("pass"); ax.set_ylabel("number of patterns"); ax.legend(fontsize=7)
ax.set_title("over-clustering and redundancy removal")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In every pass, deliberate over-clustering generated new candidate patterns; SFSC ranked the
growing library, the greedy step kept a disjoint set, and the Wilcoxon test removed patterns whose members were
explained just as well by a better one. After a few passes the selection stopped changing. Now read the composition bars
(the answer key MPP never saw):

- the **abundant ribosome** became one clean pattern holding all of its copies;
- the **TRiC pair** was found, but **split into two patterns** — the redundancy test is conservative, and two
  sub-populations that the fast 2D alignment did not bring into exactly one frame survive as separate patterns (a real
  MPP run would show both, and a person would merge them after looking);
- the **rare Hsp70 domain** was *not* found on its own: its 15 copies were absorbed into the RuBisCO pattern — at
  this resolution both are small, featureless blobs.

That is the paper's caveat in miniature: template-free mining finds the big, abundant complexes first; rare and small
ones are the hardest. (The exact splits can change with the random seed; the pattern of successes and failures is
typical.)

### 📄 Paper connection — Xu *et al.* (2019), De novo structural pattern mining (MPP)

- **Paper**: Xu M, Singla J, Tocheva EI, Chang Y-W, Stevens RC, Jensen GJ, Alber F. *De novo structural pattern mining in
  cellular electron cryotomograms.* Structure 27(4):679–691.e14 (2019).
  [doi:10.1016/j.str.2019.01.005](https://doi.org/10.1016/j.str.2019.01.005). Preprint (different title and author
  list): Xu, Tocheva, Chang, Jensen & Alber, *De novo visual proteomics in single cells through pattern mining*,
  [arXiv:1512.09347](https://arxiv.org/abs/1512.09347) (2015). Highlighted in *Nature Methods* 16:285 (2019) as
  "Template-free visual proteomics".
- **Question it answers**: can all abundant recurring structures in cellular tomograms be found without templates and
  without assuming the number of classes or well-separated classes?
- **What the paper actually does**: the loop of Section 3.2: DoG picking and optional pose normalisation; pattern
  generation by k-means on imputed, aligned subtomograms (about 10,000 informative voxels selected by average
  neighbour covariance, weighted EM-PCA to about 50 dimensions), plus sequential expansion of patterns cut where SFSC
  is maximal (and an optional genetic-algorithm step at the last iteration); SFSC from SSNR with missing-wedge counts;
  a growing pattern library; greedy disjoint selection; fast alignment of every subtomogram to every pattern;
  Wilcoxon signed-rank redundancy removal (significance 0.01); optional reference-guided level-set masking for crowded
  data; termination when the selected patterns stop changing (no new pattern for 5 passes).
- **What our toy keeps**: pose normalisation, imputation, covariance-based voxel selection, PCA + k-means
  over-clustering, SFSC from SSNR, the library, greedy disjoint selection, align-everything-to-everything, the Wilcoxon
  redundancy test (following the logic of AITom's `cluster_removal_according_to_center_matching_specificity`) and
  iteration to stability.
- **What our toy leaves out**: 3D, thousands of particles, EM-PCA with missing data, sequential expansion, the genetic
  algorithm, level-set masking, parallel computing; and our flatland SNRs are not comparable to the paper's.
- **Result reported** (arXiv v3; the journal version may differ): on 11,230 simulated subtomograms of 22 complexes,
  12 patterns after 32 iterations, 8 matching unique complexes at a false-discovery rate ≤ 10 % (4 at 0 %), with
  complexes below about 200–300 copies often missed; on about 30,000 experimental subtomograms, 12 patterns in
  *Acetonema longum*, 15 in *Hylemonella gracilis* and 10 in *Bdellovibrio bacteriovorus* (Section 4).
- **Code**: AITom `aitom/tomominer/pursuit/multi/main.py` (driver) and `.../util.py`; SSNR/FSC in
  `aitom/tomominer/statistics/ssnr.py` (`ssnr_to_fsc(ssnr) = ssnr/(2+ssnr)`); original code at
  <https://github.com/alberlab/tomominer>.

### 📄 Paper connection — Frazier, Xu & Alber (2017), TomoMiner

- **Paper**: Frazier Z, Xu M, Alber F. *TomoMiner and TomoMinerCloud: a software platform for large-scale subtomogram
  structural analysis.* Structure 25(6):951–961.e2 (2017). [doi:10.1016/j.str.2017.04.016](https://doi.org/10.1016/j.str.2017.04.016)
- ⚠️ We could not access the full text of this paper while writing; the description below is limited to its title,
  metadata and the released code. Read the paper before relying on details.
- **What it is**: the platform that runs MPP at scale. Its code (GPL-3.0; an adapted copy lives in AITom's
  `aitom/tomominer/`) contains a C++ core for fast rotational search, rotation, spherical harmonics and FFTs; Python
  modules for alignment, averaging, SSNR/FSC, EM-PCA, Multi-Pattern Pursuit, segmentation and template search; and a
  task-queue system (a queue server, a master that submits chunked tasks, many workers that execute them; Module 13).
  TomoMinerCloud, the cloud-deployable version, could not be checked.

---
## 4 · What pattern mining bought biology

The point of de novo mining is to look at cells without prejudice. In the experimental part of the MPP paper (as read
in the arXiv v3 preprint; check the journal version for final numbers), about 30,000 subtomograms from three bacteria
were mined:

| Organism | Sample | Patterns found | Notable patterns |
|---|---|---|---|
| *Acetonema longum* | lysed cell | 12 | a GroEL-like pattern (481 instances, 4.5 nm); membrane-associated patterns |
| *Hylemonella gracilis* | intact cell | 15 | a ribosome-like pattern with 802 copies |
| *Bdellovibrio bacteriovorus* | intact cell | 10 | ribosome-like patterns **depleted from a central, nucleoid-like region** |

The last row is a small example of what visual proteomics promises: not just *which* complexes exist, but **where** they
are. Ribosomes being excluded from the region occupied by the bacterial chromosome is a spatial-organisation finding
that needs no template and no labels, only patterns and their coordinates. Module 33 turns such coordinates into
statistics (nearest-neighbour distances, clustering, co-localisation), and Module 22 shows how deep clustering (DISCA)
scaled the same idea to about 200,000 subtomograms.

---
## ✅ Checkpoint

1. Why can a template-free method discover a complex that template matching would miss, and what does it pay for that?
   <details><summary>Answer</summary>It groups particles by their own recurring shapes, so no prior structure is needed; the price is that it must decide by itself what is "the same" and how many patterns exist, it can produce mixtures or artefacts, and it struggles with rare classes.</details>
2. What makes the per-shell numbers $s_{jl}$ rotation invariant?
   <details><summary>Answer</summary>A rotation mixes the spherical-harmonic coefficients only within each degree l and preserves their total energy Σ_m|a_lm|², so s_jl is unchanged.</details>
3. In the 2011 MRF, what do feature-space neighbours add that grid neighbours cannot?
   <details><summary>Answer</summary>They connect similar voxels in different, distant copies of a complex, so correctly labelled copies can pull a mislabelled copy toward the right class; grid neighbours only smooth within one object, where a wrong label is self-consistent.</details>
4. A pattern has SSNR $\eta_r=2$ in some shell. What FSC would two half-sets show there?
   <details><summary>Answer</summary>ρ = η/(2+η) = 2/4 = 0.5.</details>
5. Why does MPP deliberately over-cluster, and which step cleans up afterwards?
   <details><summary>Answer</summary>Over-clustering avoids averaging two different structures together (mixtures are hard to undo); the Wilcoxon signed-rank redundancy test then removes patterns whose members do not significantly prefer them over a better-scoring pattern.</details>
6. Why are complexes with few copies often missed by MPP?
   <details><summary>Answer</summary>SFSC grows with the number of members, so a small pure pattern scores lower than large patterns and loses in greedy selection; its members may also be absorbed by larger patterns.</details>

## 🧠 Questions & discussion

1. The 2011 method detects objects by local maxima of smoothed density. Which biological objects would this miss or
   break (think of filaments, membranes, densely packed lattices)?
2. Rotation-invariant features throw away orientation. When is orientation itself biologically interesting, and how
   could you recover it after classification?
3. In MPP the redundancy test compares alignment scores of the same members against two patterns. Why is a *paired*
   test (Wilcoxon signed-rank) appropriate here rather than an unpaired one?
4. A pattern found by MPP might be a mixture of two similar complexes. Propose two independent checks (computational or
   biochemical) that could reveal this.
5. The *B. bacteriovorus* ribosome-depletion finding relies on pattern coordinates. What artefacts in picking or
   tomogram thickness could mimic such a depletion, and how would you rule them out?

## 🚀 Bonus — going further

- **Kelley penalty.** Replace the silhouette cut in 2.4 by the Kelley *et al.* (1996) penalty used in the paper and compare.
- **More invariants.** Add degree $l=3$ harmonics (or more shells) to the features in 2.3 and test whether object
  clustering survives lower SNR.
- **Sequential expansion.** Implement MPP's step 1b: rank a pattern's candidate members by alignment score and keep the
  top-$n$ prefix that maximises SFSC.
- **Real code.** Read `aitom/tomominer/pursuit/multi/main.py` and map each block to the steps in 3.2.
- **Deep version.** Module 22 (DISCA) replaces hand-made features and k-means by a CNN trained in the loop.

## 📝 Lab note

```text
Module 15 — date:
Input: a simulated 3D tomogram (ribosome, proteasome, RuBisCO); flatland particles of 4 molecules, unequal abundance
Output: candidate objects, invariant voxel features, object classes, MRF repair; SFSC curves; mini-MPP patterns
Method: local maxima + watershed; shell spherical-harmonic energies; word histograms + hierarchical clustering;
        Potts MRF (grid + feature neighbours, ICM); SSNR → ρ = η/(2+η) → SFSC; greedy selection; Wilcoxon test
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Xu, Beck & Alber 2011; Xu et al. 2019; Wu et al. 2019; Frazier et al. 2017
```

## 📚 References

1. Xu M, Beck M, Alber F. Template-free detection of macromolecular complexes in cryo electron tomograms. *Bioinformatics* 27:i69–i76 (2011). doi:10.1093/bioinformatics/btr207
2. Xu M, Singla J, Tocheva EI, Chang Y-W, Stevens RC, Jensen GJ, Alber F. De novo structural pattern mining in cellular electron cryotomograms. *Structure* 27:679–691.e14 (2019). doi:10.1016/j.str.2019.01.005 (preprint: arXiv:1512.09347)
3. Wu X, Zeng X, Zhu Z, Gao X, Xu M. Template-based and template-free approaches in cellular cryo-electron tomography structural pattern mining. In *Computational Biology*, Codon Publications, ch. 11, pp. 175–186 (2019). doi:10.15586/computationalbiology.2019.ch11
4. Frazier Z, Xu M, Alber F. TomoMiner and TomoMinerCloud: a software platform for large-scale subtomogram structural analysis. *Structure* 25:951–961.e2 (2017). doi:10.1016/j.str.2017.04.016
5. Xu M, Zhang S, Alber F. 3D rotation invariant features for the characterization of molecular density maps. *IEEE BIBM* 2009, pp. 74–78. doi:10.1109/BIBM.2009.32
6. Kelley LA, Gardner SP, Sutcliffe MJ. An automated approach for clustering an ensemble of NMR-derived protein structures into conformationally related subfamilies. *Protein Eng* 9:1063–1065 (1996). doi:10.1093/protein/9.11.1063

## ➡️ Next

**[16 · 3D convolutional networks for subtomogram classification](16_3d_cnn_classification.ipynb)**: Part D begins:
learned features instead of hand-made ones. Course overview: [`README.md`](../README.md).